# 放物運動シミュレーション（ローカル）

Cursor 上では `ipywidgets` のスライダーが表示されないため、matplotlib のスライダーを使います。
実行すると図ウィンドウが開き、下のスライダーで初速度・角度・軸範囲を変えられます。
**カーネルを再起動してから、上から順に実行してください。**

In [1]:
%matplotlib tk
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
from matplotlib import font_manager

available = {f.name for f in font_manager.fontManager.ttflist}
for name in ("Yu Gothic", "YuGothic", "Meiryo", "MS Gothic"):
    if name in available:
        plt.rcParams["font.family"] = name
        break
plt.rcParams["axes.unicode_minus"] = False

In [2]:
G = 9.8  # 地球上の重力加速度 [m/s²]


def compute(v0, angle):
    theta = np.radians(angle)
    vx = v0 * np.cos(theta)
    vy = v0 * np.sin(theta)
    flight_time = 2 * vy / G if vy > 0 else 0.1
    t = np.linspace(0, flight_time, 200)
    x = vx * t
    y = vy * t - 0.5 * G * t**2
    max_height = vy**2 / (2 * G)
    distance = float(x[-1])
    return x, y, flight_time, max_height, distance

In [3]:
fig, ax = plt.subplots(figsize=(8, 6))
plt.subplots_adjust(bottom=0.32)

x, y, flight_time, max_height, distance = compute(15.0, 45.0)
line, = ax.plot(x, y, lw=2)
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("放物運動")
ax.grid(True)
ax.set_xlim(0, 100)
ax.set_ylim(0, 50)
info = ax.text(
    0.02,
    0.98,
    f"飛行時間 : {flight_time:.2f} s\n最高点 : {max_height:.2f} m\n水平到達距離: {distance:.2f} m",
    transform=ax.transAxes,
    va="top",
)

ax_v0 = fig.add_axes([0.18, 0.20, 0.65, 0.03])
ax_ang = fig.add_axes([0.18, 0.15, 0.65, 0.03])
ax_xm = fig.add_axes([0.18, 0.10, 0.65, 0.03])
ax_ym = fig.add_axes([0.18, 0.05, 0.65, 0.03])

s_v0 = Slider(ax_v0, "初速度 [m/s]", 1.0, 30.0, valinit=15.0, valstep=1.0)
s_ang = Slider(ax_ang, "角度 [°]", 0.0, 90.0, valinit=45.0, valstep=1.0)
s_xm = Slider(ax_xm, "X軸上限 [m]", 10.0, 200.0, valinit=100.0, valstep=5.0)
s_ym = Slider(ax_ym, "Y軸上限 [m]", 5.0, 100.0, valinit=50.0, valstep=5.0)


def update(_=None):
    x, y, flight_time, max_height, distance = compute(s_v0.val, s_ang.val)
    line.set_data(x, y)
    ax.set_xlim(0, s_xm.val)
    ax.set_ylim(0, s_ym.val)
    info.set_text(
        f"飛行時間 : {flight_time:.2f} s\n最高点 : {max_height:.2f} m\n水平到達距離: {distance:.2f} m"
    )
    fig.canvas.draw_idle()


s_v0.on_changed(update)
s_ang.on_changed(update)
s_xm.on_changed(update)
s_ym.on_changed(update)
plt.show(block=False)